# Linear probing on VTAB-1k (RepAdapter bundle) from a MoCo v3 ViT-Tiny/8 checkpoint

Same protocol as [cifar-finetuning.ipynb](cifar-finetuning.ipynb): freeze the MoCo
backbone, extract pre-logit features once, and train a fresh per-task linear
classifier on the cached features (SGD, momentum 0.9, weight-decay 0, cosine LR,
base LR 0.1 scaled by `batch_size/256`).

**Data:** the RepAdapter **VTAB-1k** bundle under `vtab-1k/`. For every task the
linear head is trained on the **`train800val200`** split (1000 images) and evaluated
on the full **`test`** split — the canonical VTAB-1k protocol.

**Tasks (this part):**

Images are resized to 64x64 to match the backbone (patch_size=8 -> 8x8 = 64 tokens).
These are VTAB-1k numbers (1000 training images), so they track this model but are
not directly comparable to published 224px results.

## 1. Imports

In [1]:
import math
import os
import random
import time

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torch.optim
import torch.utils.data
import torchvision.transforms as transforms
from torchvision.datasets.folder import default_loader

from moco_tiny.vits import vit_tiny

/opt/anaconda3/envs/visual-ai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

Set `pretrained` to your MoCo v3 ViT-Tiny/8 checkpoint. `data_root` points at the
RepAdapter VTAB-1k bundle. The `VTAB_TASKS` registry (bundle folder + class count)
is the only per-task variation -- everything downstream is task-agnostic.

In [2]:
class Config:
    # path to a MoCo v3 ViT-Tiny/8 checkpoint
    pretrained = '/Users/kulmak41/Library/CloudStorage/GoogleDrive-kulmak41@gmail.com/My Drive/vast-data/in_batch/100k_4000_batch/checkpoint_0281.pth.tar'

    # RepAdapter VTAB-1k bundle root
    data_root = 'vtab-1k'
    train_split = 'train800val200'   # 1000 images
    test_split = 'test'

    # must match the pretrained backbone (patch_size=8, grid 8x8 = 64 tokens)
    image_size = 64

    # linear probing recipe -- main_lincls.py defaults
    epochs = 90
    batch_size = 256
    lr = 0.1            # base LR; scaled by batch_size/256
    momentum = 0.9
    weight_decay = 0.0
    workers = 2

    seed = 1205


args = Config()

# VTAB-1k task folders (the bundle dir name doubles as the label). num_classes is
# auto-derived from the labels. All 19 VTAB tasks, grouped by category:
VTAB_TASKS = [
    # natural
    'caltech101', 'cifar', 'dtd', 'oxford_flowers102', 'oxford_iiit_pet', 'sun397', 'svhn',
    # specialized
    'eurosat', 'patch_camelyon', 'resisc45', 'diabetic_retinopathy',
    # structured
    'clevr_count', 'clevr_dist', 'dmlab', 'kitti',
    'dsprites_loc', 'dsprites_ori', 'smallnorb_azi', 'smallnorb_ele',
]

if args.seed is not None:
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    cudnn.deterministic = True

## 3. Build model and load the MoCo checkpoint (shared frozen backbone)

The backbone is frozen and we read **pre-logit** features via `forward_features` ->
`forward_head(pre_logits=True)` (CLS-token pool + final norm, stops before the
classifier). So `num_classes` here is irrelevant and the *same* frozen backbone
serves every task. Loading follows the rename/strip logic in
[main_lincls.py:177-189](main_lincls.py#L177-L189).

In [3]:
# pick the best available device -- CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'=> using device: {device}')

LINEAR_KEYWORD = 'head'

print("=> creating model 'vit_tiny/8'")
# num_classes is unused: we read pre-logit features, so the head is never applied.
model = vit_tiny(img_size=args.image_size, num_classes=10)

# freeze the entire backbone
for _, param in model.named_parameters():
    param.requires_grad = False

# load MoCo pretrained backbone
assert os.path.isfile(args.pretrained), f"no checkpoint at '{args.pretrained}'"
print(f"=> loading checkpoint '{args.pretrained}'")
checkpoint = torch.load(args.pretrained, map_location='cpu')

state_dict = checkpoint['state_dict']
renamed = {}
for k, v in state_dict.items():
    # single-GPU pretraining: keys start with 'base_encoder.' (no 'module.'); handle both
    key = k[len('module.'):] if k.startswith('module.') else k
    if key.startswith('base_encoder.') and not key.startswith(f'base_encoder.{LINEAR_KEYWORD}.'):
        renamed[key[len('base_encoder.'):]] = v

msg = model.load_state_dict(renamed, strict=False)
print(f'   missing keys:    {msg.missing_keys}')
print(f'   unexpected keys: {msg.unexpected_keys}')
assert set(msg.missing_keys) <= {f'{LINEAR_KEYWORD}.weight', f'{LINEAR_KEYWORD}.bias'}, f'unexpected missing keys: {msg.missing_keys}'
assert not msg.unexpected_keys, f'unexpected keys: {msg.unexpected_keys}'
print(f"=> loaded pre-trained backbone (pretraining epoch {checkpoint.get('epoch', '?')})")

model = model.to(device)
model.eval()
cudnn.benchmark = True

=> using device: mps
=> creating model 'vit_tiny/8'
=> loading checkpoint '/Users/kulmak41/Library/CloudStorage/GoogleDrive-kulmak41@gmail.com/My Drive/vast-data/in_batch/100k_4000_batch/checkpoint_0281.pth.tar'
   missing keys:    ['head.weight', 'head.bias']
   unexpected keys: []
=> loaded pre-trained backbone (pretraining epoch 282)


## 4. VTAB-1k data loader (disk-backed)

Each task folder holds split lists `train800val200.txt` / `test.txt` whose lines are
`images/<split>/NNNNNN.jpg <label>`. `VtabImageDataset` keeps only the `(path, label)`
list and loads each image lazily in `__getitem__` via torchvision's `default_loader`
(open + `convert('RGB')`) -- the disk-backed, ImageFolder-style replacement for the old
in-RAM `CachedMoCoDataset`, so no decoded images are kept in memory. `num_classes` is
derived from the labels (max over train+test) instead of hardcoded, and the train-side
split sizes are sanity-checked on load.

In [4]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

eval_transform = transforms.Compose([
    transforms.Resize((args.image_size, args.image_size)),
    transforms.ToTensor(),
    normalize,
])

# expected sizes for the VTAB-1k train-side splits (cheap integrity check)
_SPLIT_SIZES = {'train800': 800, 'val200': 200, 'train800val200': 1000}


class VtabImageDataset(torch.utils.data.Dataset):
    """Disk-backed VTAB-1k split. Stores only the (image path, label) list and
    loads + transforms each image lazily in __getitem__ via torchvision's
    default_loader -- the ImageFolder pattern, driven by the bundle's
    `<split>.txt` label file (the bundle has no class-named subdirs). No decoded
    images are held in RAM; the DataLoader streams them per batch."""

    def __init__(self, root, task_dir, split, transform):
        base = os.path.join(root, task_dir)
        self.transform = transform
        with open(os.path.join(base, f'{split}.txt')) as f:
            lines = f.read().splitlines()
        self.samples = [(os.path.join(base, rel), int(lab))
                        for rel, lab in (ln.split() for ln in lines)]
        if split in _SPLIT_SIZES:
            assert len(self.samples) == _SPLIT_SIZES[split], f"{task_dir}/{split}: expected {_SPLIT_SIZES[split]} images, got {len(self.samples)}"

    @property
    def num_classes(self):
        return max(lab for _, lab in self.samples) + 1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(default_loader(path)), label

## 5. Helpers -- meters, accuracy, LR schedule, feature extraction, cached training

Copied from [cifar-finetuning.ipynb](cifar-finetuning.ipynb). `iterate_batches` keeps
every sample (no `drop_last`) since the train split is only 1000 images.

In [5]:
class AverageMeter(object):
    def __init__(self, name, fmt=':f'):
        self.name = name; self.fmt = fmt; self.reset()
    def reset(self):
        self.val = 0; self.avg = 0; self.sum = 0; self.count = 0
    def update(self, val, n=1):
        self.val = val; self.sum += val * n; self.count += n; self.avg = self.sum / self.count


def accuracy(output, target, topk=(1,)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        return [correct[:k].reshape(-1).float().sum(0, keepdim=True).mul_(100.0 / batch_size)
                for k in topk]


def adjust_learning_rate(optimizer, init_lr, epoch, total_epochs):
    cur_lr = init_lr * 0.5 * (1. + math.cos(math.pi * epoch / total_epochs))
    for param_group in optimizer.param_groups:
        param_group['lr'] = cur_lr
    return cur_lr


@torch.no_grad()
def extract_features(loader, model, device):
    model.eval()
    feats, labels = [], []
    for images, target in loader:
        images = images.to(device, non_blocking=True)
        tokens = model.forward_features(images)
        f = model.forward_head(tokens, pre_logits=True)
        feats.append(f.cpu()); labels.append(target)
    return torch.cat(feats), torch.cat(labels)


def iterate_batches(feats, labels, batch_size, shuffle):
    n = feats.shape[0]
    idx = torch.randperm(n, device=feats.device) if shuffle else torch.arange(n, device=feats.device)
    for start in range(0, n, batch_size):   # keep all samples (1000-image train split)
        sel = idx[start:start + batch_size]
        yield feats[sel], labels[sel]


@torch.no_grad()
def evaluate_cached(classifier, criterion, feats, labels, batch_size):
    classifier.eval()
    losses = AverageMeter('Loss'); top1 = AverageMeter('Acc@1')
    for x, y in iterate_batches(feats, labels, batch_size, shuffle=False):
        logits = classifier(x)
        loss = criterion(logits, y)
        (a1,) = accuracy(logits, y, topk=(1,))
        losses.update(loss.item(), x.size(0)); top1.update(a1.item(), x.size(0))
    return losses.avg, top1.avg

## 6. Run linear probing on each task (train800val200 -> test)

For each task: load the train/test splits from the bundle, extract features through
the shared frozen backbone, then train a fresh `nn.Linear(192, num_classes)` on the
cache. Per-task and mean top-1 are collected in `results`.

In [6]:
def run_task(task, model, device, args):
    tr_ds = VtabImageDataset(args.data_root, task, args.train_split, eval_transform)
    te_ds = VtabImageDataset(args.data_root, task, args.test_split, eval_transform)
    num_classes = max(tr_ds.num_classes, te_ds.num_classes)
    print(f"\n===== {task}  ({num_classes} classes) =====")
    print(f'   {len(tr_ds)} train + {len(te_ds)} test images (disk-backed, lazy load)')

    # num_workers=0: VtabImageDataset is defined in this notebook, so spawned
    # DataLoader workers on macOS can't import it to unpickle. Feature extraction
    # is a single pass, so in-process disk loading is fine here. (To use workers,
    # move VtabImageDataset into moco_tiny/loader.py like CachedMoCoDataset.)
    pin = (device.type == 'cuda')
    tr_loader = torch.utils.data.DataLoader(tr_ds, batch_size=args.batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    te_loader = torch.utils.data.DataLoader(te_ds, batch_size=args.batch_size, shuffle=False, num_workers=0, pin_memory=pin)

    # one-shot feature extraction through the shared frozen backbone
    train_feats, train_labels = extract_features(tr_loader, model, device)
    test_feats,  test_labels  = extract_features(te_loader, model, device)
    train_feats = train_feats.to(device); train_labels = train_labels.to(device)
    test_feats  = test_feats.to(device);  test_labels  = test_labels.to(device)

    # fresh linear classifier on the cached features
    embed_dim = train_feats.shape[1]
    classifier = nn.Linear(embed_dim, num_classes).to(device)
    classifier.weight.data.normal_(mean=0.0, std=0.01)
    classifier.bias.data.zero_()
    criterion = nn.CrossEntropyLoss().to(device)

    init_lr = args.lr * args.batch_size / 256
    optimizer = torch.optim.SGD(classifier.parameters(), init_lr, momentum=args.momentum, weight_decay=args.weight_decay)

    hist = dict(train_loss=[], train_acc=[], test_acc=[])
    best_acc1 = 0.0
    for epoch in range(args.epochs):
        cur_lr = adjust_learning_rate(optimizer, init_lr, epoch, args.epochs)
        classifier.train()
        losses = AverageMeter('Loss'); top1 = AverageMeter('Acc@1')
        for x, y in iterate_batches(train_feats, train_labels, args.batch_size, shuffle=True):
            logits = classifier(x); loss = criterion(logits, y)
            (a1,) = accuracy(logits, y, topk=(1,))
            losses.update(loss.item(), x.size(0)); top1.update(a1.item(), x.size(0))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        te_loss, te_acc1 = evaluate_cached(classifier, criterion, test_feats, test_labels, args.batch_size)
        hist['train_loss'].append(losses.avg); hist['train_acc'].append(top1.avg)
        hist['test_acc'].append(te_acc1)
        best_acc1 = max(best_acc1, te_acc1)
        if epoch % 10 == 0 or epoch == args.epochs - 1:
            print(f'   epoch {epoch:3d}  lr={cur_lr:.4f}  train@1={top1.avg:6.2f}  test@1={te_acc1:6.2f}')
    print(f'   => {task}: best test top-1 = {best_acc1:.2f}%  (final {hist["test_acc"][-1]:.2f}%)')
    return dict(best_acc1=best_acc1, final_acc1=hist['test_acc'][-1], num_classes=num_classes, hist=hist)


results = {}
overall_t0 = time.time()
for _dir in VTAB_TASKS:
    results[_dir] = run_task(_dir, model, device, args)
print(f'\n=> all {len(results)} tasks done in {time.time()-overall_t0:.1f}s')


===== caltech101  (102 classes) =====
   1000 train + 6085 test images (disk-backed, lazy load)
   epoch   0  lr=0.1000  train@1=  0.90  test@1=  0.12
   epoch  10  lr=0.0970  train@1= 16.40  test@1=  6.10
   epoch  20  lr=0.0883  train@1= 28.00  test@1= 15.45
   epoch  30  lr=0.0750  train@1= 39.10  test@1= 26.10
   epoch  40  lr=0.0587  train@1= 43.90  test@1= 27.12
   epoch  50  lr=0.0413  train@1= 47.40  test@1= 30.44
   epoch  60  lr=0.0250  train@1= 49.50  test@1= 31.98
   epoch  70  lr=0.0117  train@1= 50.70  test@1= 32.21
   epoch  80  lr=0.0030  train@1= 51.20  test@1= 32.49
   epoch  89  lr=0.0000  train@1= 51.40  test@1= 32.54
   => caltech101: best test top-1 = 32.54%  (final 32.54%)

===== cifar  (100 classes) =====
   1000 train + 10000 test images (disk-backed, lazy load)
   epoch   0  lr=0.1000  train@1=  1.20  test@1=  1.12
   epoch  10  lr=0.0970  train@1= 11.20  test@1=  5.32
   epoch  20  lr=0.0883  train@1= 18.10  test@1=  8.24
   epoch  30  lr=0.0750  train@1= 24

## 7. Summary

In [7]:
print(f'{"task":<20}{"classes":>8}{"best@1":>9}{"final@1":>9}')
print('-' * 46)
for name, r in results.items():
    print(f'{name:<20}{r["num_classes"]:>8}{r["best_acc1"]:>9.2f}{r["final_acc1"]:>9.2f}')
mean_best = sum(r['best_acc1'] for r in results.values()) / len(results)
mean_final = sum(r['final_acc1'] for r in results.values()) / len(results)
print('-' * 46)
print(f'{"mean":<20}{"":>8}{mean_best:>9.2f}{mean_final:>9.2f}')

task                 classes   best@1  final@1
----------------------------------------------
caltech101               102    32.54    32.54
cifar                    100    13.45    13.43
dtd                       47    27.50    27.45
oxford_flowers102        102    23.69    23.69
oxford_iiit_pet           37    22.89    22.84
sun397                   397     5.54     5.54
svhn                      10    32.86    32.76
eurosat                   10    81.35    81.31
patch_camelyon             2    81.47    81.29
resisc45                  45    35.92    35.90
diabetic_retinopathy       5    73.60    73.59
clevr_count                8    33.22    33.03
clevr_dist                 6    37.36    36.68
dmlab                      6    33.16    33.07
kitti                      4    50.77    48.95
dsprites_loc              16    14.19    14.13
dsprites_ori              16    13.36    13.32
smallnorb_azi             18    11.68    10.83
smallnorb_ele              9    19.39    19.35
-------------

## 8. Curves

In [ ]:
import matplotlib.pyplot as plt

n = len(results)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), squeeze=False)
for ax, (name, r) in zip(axes[0], results.items()):
    ep = range(len(r['hist']['test_acc']))
    ax.plot(ep, r['hist']['train_acc'], label='train@1')
    ax.plot(ep, r['hist']['test_acc'],  label='test@1')
    ax.set_title(f'{name} (best {r["best_acc1"]:.1f}%)')
    ax.set_xlabel('epoch'); ax.set_ylabel('accuracy (%)')
    ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()